In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.stattools import adfuller

# ============================================================
# ADF Test
# - input:
#   1) ./tvpvar_raw_level_merged.csv
#   2) ./tvpvar_input_scaled.csv
# - output:
#   ./result/adf_raw.csv
#   ./result/adf_transformed.csv
#   ./result/adf_summary.txt
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")
RESULT_DIR = BASE_DIR / "result"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = BASE_DIR / "tvpvar_raw_level_merged.csv"
TRANS_FILE = BASE_DIR / "tvpvar_input_scaled.csv"

OUT_RAW = RESULT_DIR / "adf_raw.csv"
OUT_TRANS = RESULT_DIR / "adf_transformed.csv"
OUT_SUMMARY = RESULT_DIR / "adf_summary.txt"

DATE_CANDIDATES = ["Date", "date", "DATE"]


# -----------------------------
# Helpers
# -----------------------------
def find_date_col(df):
    for c in DATE_CANDIDATES:
        if c in df.columns:
            return c
    return None


def load_csv_complete_case(path):
    if not path.exists():
        raise FileNotFoundError(f"파일을 찾을 수 없습니다: {path}")

    df = pd.read_csv(path)

    date_col = find_date_col(df)
    if date_col is not None:
        df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

    before = len(df)

    df = df.replace([np.inf, -np.inf], np.nan)

    # 하나라도 결측이면 해당 row 전체 제거
    df = df.dropna(how="any").copy()

    if date_col is not None:
        df = df.sort_values(date_col).reset_index(drop=True)

    after = len(df)

    print(f"Loaded: {path}")
    print(f"Rows before dropna: {before}")
    print(f"Rows after dropna : {after}")
    print(f"Dropped rows      : {before - after}")
    print()

    return df


def get_numeric_columns(df):
    return [
        col for col in df.columns
        if pd.api.types.is_numeric_dtype(df[col])
    ]


def run_adf(series, var_name, regression="c", autolag="AIC"):
    s = (
        pd.to_numeric(series, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    result = {
        "variable": var_name,
        "n_obs": int(len(s)),
        "regression": regression,
        "autolag": autolag,
        "adf_stat": np.nan,
        "p_value": np.nan,
        "used_lag": np.nan,
        "n_used_for_regression": np.nan,
        "crit_1%": np.nan,
        "crit_5%": np.nan,
        "crit_10%": np.nan,
        "is_stationary_5pct": np.nan,
        "status": "",
    }

    if len(s) < 20:
        result["status"] = "too_few_observations"
        return result

    if s.nunique() <= 1:
        result["status"] = "constant_or_single_value"
        return result

    try:
        adf_res = adfuller(s, regression=regression, autolag=autolag)

        result["adf_stat"] = adf_res[0]
        result["p_value"] = adf_res[1]
        result["used_lag"] = adf_res[2]
        result["n_used_for_regression"] = adf_res[3]
        result["crit_1%"] = adf_res[4].get("1%", np.nan)
        result["crit_5%"] = adf_res[4].get("5%", np.nan)
        result["crit_10%"] = adf_res[4].get("10%", np.nan)
        result["is_stationary_5pct"] = bool(adf_res[1] < 0.05)
        result["status"] = "ok"

    except Exception as e:
        result["status"] = f"error: {str(e)}"

    return result


def adf_table(df, dataset_name, regression="c", autolag="AIC"):
    num_cols = get_numeric_columns(df)
    rows = []

    for col in num_cols:
        res = run_adf(
            series=df[col],
            var_name=col,
            regression=regression,
            autolag=autolag,
        )
        res["dataset"] = dataset_name
        rows.append(res)

    out = pd.DataFrame(rows)

    if not out.empty:
        out = out[
            [
                "dataset",
                "variable",
                "n_obs",
                "regression",
                "autolag",
                "adf_stat",
                "p_value",
                "used_lag",
                "n_used_for_regression",
                "crit_1%",
                "crit_5%",
                "crit_10%",
                "is_stationary_5pct",
                "status",
            ]
        ]

    return out


def format_summary(raw_res, trans_res):
    lines = []

    lines.append("============================================================")
    lines.append("ADF Unit Root Test Summary")
    lines.append("============================================================")
    lines.append("")

    def add_block(title, df_res):
        lines.append(title)
        lines.append("-" * len(title))

        if df_res.empty:
            lines.append("No results.")
            lines.append("")
            return

        ok = df_res[df_res["status"] == "ok"].copy()
        fail = df_res[df_res["status"] != "ok"].copy()

        if not ok.empty:
            lines.append("Variables tested:")
            for _, row in ok.iterrows():
                lines.append(
                    f"  - {row['variable']}: "
                    f"ADF={row['adf_stat']:.6f}, "
                    f"p={row['p_value']:.6f}, "
                    f"lag={int(row['used_lag']) if pd.notna(row['used_lag']) else 'NA'}, "
                    f"stationary@5%={row['is_stationary_5pct']}"
                )
        else:
            lines.append("No successful ADF results.")

        if not fail.empty:
            lines.append("")
            lines.append("Failed / skipped variables:")
            for _, row in fail.iterrows():
                lines.append(f"  - {row['variable']}: {row['status']}")

        lines.append("")

    add_block("1) Raw level data", raw_res)
    add_block("2) Transformed / scaled data", trans_res)

    ok_trans = trans_res[trans_res["status"] == "ok"].copy()

    stationary_cols = ok_trans.loc[
        ok_trans["is_stationary_5pct"] == True,
        "variable"
    ].tolist()

    non_stationary_cols = ok_trans.loc[
        ok_trans["is_stationary_5pct"] == False,
        "variable"
    ].tolist()

    lines.append("3) Final check for transformed / scaled data")
    lines.append("-------------------------------------------")
    lines.append(f"Stationary @ 5%: {stationary_cols if stationary_cols else 'None'}")
    lines.append(f"Not stationary @ 5%: {non_stationary_cols if non_stationary_cols else 'None'}")
    lines.append("")

    return "\n".join(lines)


# -----------------------------
# Main
# -----------------------------
def main():
    print("[1/4] Loading complete-case data...")

    df_raw = load_csv_complete_case(RAW_FILE)
    df_trans = load_csv_complete_case(TRANS_FILE)

    print("Raw shape        :", df_raw.shape)
    print("Transformed shape:", df_trans.shape)

    print("\n[2/4] Running ADF on raw level data...")
    raw_res = adf_table(
        df_raw,
        dataset_name="raw_level",
        regression="c",
        autolag="AIC",
    )

    print("[3/4] Running ADF on transformed / scaled data...")
    trans_res = adf_table(
        df_trans,
        dataset_name="transformed_scaled",
        regression="c",
        autolag="AIC",
    )

    print("\n[4/4] Saving results to ./result ...")

    raw_res.to_csv(OUT_RAW, index=False, encoding="utf-8-sig")
    trans_res.to_csv(OUT_TRANS, index=False, encoding="utf-8-sig")

    summary_text = format_summary(raw_res, trans_res)
    with open(OUT_SUMMARY, "w", encoding="utf-8") as f:
        f.write(summary_text)

    print("\nSaved:")
    print(f" - {OUT_RAW}")
    print(f" - {OUT_TRANS}")
    print(f" - {OUT_SUMMARY}")

    print("\nPreview: transformed / scaled data ADF")
    if not trans_res.empty:
        preview_cols = [
            "variable",
            "n_obs",
            "adf_stat",
            "p_value",
            "used_lag",
            "is_stationary_5pct",
            "status",
        ]
        print(trans_res[preview_cols].to_string(index=False))
    else:
        print("No transformed ADF results.")

    print("\nDone.")


if __name__ == "__main__":
    main()

[1/4] Loading complete-case data...
Loaded: tvpvar_raw_level_merged.csv
Rows before dropna: 1033
Rows after dropna : 1033
Dropped rows      : 0

Loaded: tvpvar_input_scaled.csv
Rows before dropna: 1032
Rows after dropna : 1032
Dropped rows      : 0

Raw shape        : (1033, 10)
Transformed shape: (1032, 10)

[2/4] Running ADF on raw level data...
[3/4] Running ADF on transformed / scaled data...

[4/4] Saving results to ./result ...

Saved:
 - result\adf_raw.csv
 - result\adf_transformed.csv
 - result\adf_summary.txt

Preview: transformed / scaled data ADF
   variable  n_obs   adf_stat      p_value  used_lag  is_stationary_5pct status
dlog_SOLVPN   1032 -30.456048 0.000000e+00         0                True     ok
dlog_COPPER   1032 -32.902080 0.000000e+00         0                True     ok
  dlog_GOLD   1032  -6.075813 1.122336e-07        21                True     ok
dlog_SILVER   1032 -34.936330 0.000000e+00         0                True     ok
   dlog_DXY   1032 -24.907035 0.0000